<a href="https://colab.research.google.com/github/AlifHammam/data-science-2026/blob/main/Pertemuan6_AlifHamammamMultazam_240401010043.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 6 - Persiapan Data

**Nama Lengkap:** Alif Hamammam Multazam  
**NIM:** 240401010043  
**Kelas:** IF403

## Langkah 1 — Load & EDA Singkat

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

df = sns.load_dataset('titanic')

# Pilih kolom yang akan digunakan
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = df[cols].copy()

print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))

Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


## Materi Encoding — Label, One-Hot, dan Ordinal Encoding

In [2]:
# Label Encoding
df_enc = pd.DataFrame({
    'Gender': ['male', 'female', 'female', 'male', 'female'],
    'Survived': [0, 1, 1, 0, 1]
})

le = LabelEncoder()
df_enc['Gender_enc'] = le.fit_transform(df_enc['Gender'])
print('Label Encoding:')
print(df_enc)

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Mapping:', mapping)

decoded = le.inverse_transform([0, 1, 0])
print('Decoded:', decoded)

Label Encoding:
   Gender  Survived  Gender_enc
0    male         0           1
1  female         1           0
2  female         1           0
3    male         0           1
4  female         1           0
Mapping: {'female': np.int64(0), 'male': np.int64(1)}
Decoded: ['female' 'male' 'female']


In [3]:
# One-Hot Encoding
df_city = pd.DataFrame({'City': ['Jakarta', 'Surabaya', 'Bandung', 'Jakarta', 'Surabaya']})

# Cara 1: pandas get_dummies
df_ohe = pd.get_dummies(df_city, columns=['City'], drop_first=False, dtype=int)
print('One-Hot Encoding (get_dummies):')
print(df_ohe)

# Cara 2: sklearn OneHotEncoder
enc = OneHotEncoder(sparse_output=False, drop='first')
X_enc = enc.fit_transform(df_city[['City']])
print('\nNama fitur (sklearn):', enc.get_feature_names_out())

One-Hot Encoding (get_dummies):
   City_Bandung  City_Jakarta  City_Surabaya
0             0             1              0
1             0             0              1
2             1             0              0
3             0             1              0
4             0             0              1

Nama fitur (sklearn): ['City_Jakarta' 'City_Surabaya']


In [4]:
# Ordinal Encoding
df_edu = pd.DataFrame({
    'Pendidikan': ['SMA', 'S1', 'SD', 'D3', 'S2', 'SMP'],
    'Gaji_juta': [5, 12, 3, 8, 18, 4]
})

edu_order = [['SD', 'SMP', 'SMA', 'D3', 'S1', 'S2']]
ord_enc = OrdinalEncoder(categories=edu_order,
                         handle_unknown='use_encoded_value',
                         unknown_value=-1)

df_edu['Pendidikan_enc'] = ord_enc.fit_transform(df_edu[['Pendidikan']])
print('Ordinal Encoding:')
print(df_edu.sort_values('Pendidikan_enc'))

Ordinal Encoding:
  Pendidikan  Gaji_juta  Pendidikan_enc
2         SD          3             0.0
5        SMP          4             1.0
0        SMA          5             2.0
3         D3          8             3.0
1         S1         12             4.0
4         S2         18             5.0


## Materi Scaling — MinMaxScaler, StandardScaler, RobustScaler

In [5]:
df_scale = pd.DataFrame({
    'Usia': [25, 45, 32, 55, 28],
    'Pendapatan': [5, 20, 8, 35, 12]
})

# MinMaxScaler
mm = MinMaxScaler()
X_mm = mm.fit_transform(df_scale)
print('MinMaxScaler:')
print(pd.DataFrame(X_mm, columns=['Usia_mm', 'Pendapatan_mm']).round(3))

# StandardScaler
ss = StandardScaler()
X_ss = ss.fit_transform(df_scale)
print('\nStandardScaler:')
print(pd.DataFrame(X_ss, columns=['Usia_z', 'Pendapatan_z']).round(3))

# RobustScaler
rs = RobustScaler()
X_rs = rs.fit_transform(df_scale)
print('\nRobustScaler:')
print(pd.DataFrame(X_rs, columns=['Usia_rob', 'Pendapatan_rob']).round(3))

MinMaxScaler:
   Usia_mm  Pendapatan_mm
0    0.000          0.000
1    0.667          0.500
2    0.233          0.100
3    1.000          1.000
4    0.100          0.233

StandardScaler:
   Usia_z  Pendapatan_z
0  -1.062        -1.023
1   0.708         0.372
2  -0.443        -0.744
3   1.593         1.767
4  -0.797        -0.372

RobustScaler:
   Usia_rob  Pendapatan_rob
0    -0.412          -0.583
1     0.765           0.667
2     0.000          -0.333
3     1.353           1.917
4    -0.235           0.000


## Langkah 2 — Handling Missing Values

In [6]:
# Age: isi dengan median
df['age'] = df['age'].fillna(df['age'].median())

# Embarked: isi dengan modus
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Missing setelah handling:')
print(df.isnull().sum())

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


## Langkah 3 — Encoding Kategorikal

In [7]:
# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True, dtype=int)

print('Kolom setelah encoding:')
print(df.columns.tolist())
print(df.head(3))

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']
   pclass   age  sibsp  parch     fare  survived  sex_male  embarked_Q  \
0       3  22.0      1      0   7.2500         0         1           0   
1       1  38.0      1      0  71.2833         1         0           0   
2       3  26.0      0      0   7.9250         1         0           0   

   embarked_S  
0           1  
1           0  
2           1  


## Langkah 4 — Train-Test Split

In [8]:
X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # proporsi kelas terjaga
)

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')

print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))

print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


## Langkah 5 — Feature Scaling

In [9]:
# Hanya kolom numerik yang di-scale
# Kolom biner hasil OHE (sex_male, embarked_Q, embarked_S) tidak perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

scaler = StandardScaler()

# fit_transform pada training set — belajar μ dan σ dari sini
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform saja pada test set — gunakan μ dan σ dari training!
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train) :', scaler.scale_.round(2))

print('\nContoh X_train setelah scaling:')
print(X_train.head(3).round(3))

print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape},  y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train) : [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8),  y_test : (179,)


## Kesimpulan

Pada praktikum pertemuan keenam ini saya mempelajari tahap persiapan data sebelum model Machine Learning dapat dilatih, yang mencakup encoding data kategorikal menggunakan Label Encoding, One-Hot Encoding, dan Ordinal Encoding, serta scaling fitur numerik menggunakan MinMaxScaler, StandardScaler, dan RobustScaler.

Melalui pipeline preprocessing pada dataset Titanic, saya memahami pentingnya urutan langkah yang benar: handling missing values, encoding, train-test split, lalu baru scaling. Dari proses ini saya juga belajar konsep data leakage, yaitu kenapa scaler harus di-fit hanya pada training set dan tidak boleh pada keseluruhan data, serta pentingnya stratify saat split untuk menjaga proporsi kelas yang tidak seimbang.

Keterbatasan pada praktikum ini adalah pipeline yang dibangun masih dilakukan secara manual langkah per langkah, belum menggunakan Pipeline dari scikit-learn yang lebih efisien dan lebih aman dari risiko data leakage. Namun, praktikum ini sangat membantu dalam memahami alasan dan logika di balik setiap tahap persiapan data sebelum masuk ke proses pemodelan.